In [ ]:
# Cell 1: Import all required packages
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

print("All libraries imported successfully!")

All libraries imported successfully!


In [ ]:

# Cell 2: Generate the synthetic student dataset
np.random.seed(42)
n_samples = 500

# Feature creation based on academic metrics
attendance = np.random.normal(loc=75, scale=15, size=n_samples).clip(40, 100)
assignment_score = np.random.normal(loc=70, scale=18, size=n_samples).clip(30, 100)
quiz_score = np.random.normal(loc=65, scale=20, size=n_samples).clip(20, 100)
study_hours = np.random.normal(loc=12, scale=5, size=n_samples).clip(2, 30)

# Performance calculation formula (1 = Safe, 0 = At-Risk)
score_index = (attendance * 0.3) + (assignment_score * 0.3) + (quiz_score * 0.3) + (study_hours * 1.5)
target = np.where(score_index >= 65, 1, 0)

# Build DataFrame `df`
df = pd.DataFrame({
    'Attendance_Rate': np.round(attendance, 2),
    'Assignment_Score': np.round(assignment_score, 2),
    'Quiz_Score': np.round(quiz_score, 2),
    'Study_Hours': np.round(study_hours, 1),
    'Pass_Status': target
})

print("Dataset created successfully! First 5 rows:")
df.head()

Dataset created successfully! First 5 rows:


,Attendance_Rate,Assignment_Score,Quiz_Score,Study_Hours,Pass_Status
0,82.45,86.67,92.99,15.9,1
1,72.93,100.00,83.49,9.2,1
2,84.72,44.83,66.19,7.9,1
3,97.85,80.13,52.06,12.0,1
4,71.49,58.29,78.96,11.1,1


In [ ]:
# Cell 3: Preprocessing and scaling
X = df.drop(columns=['Pass_Status'])
y = df['Pass_Status']

# Split data: 80% train, 20% test
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Normalization using StandardScaler
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Data preprocessing complete!")
print(f"Training set shape: {X_train_scaled.shape}")
print(f"Testing set shape: {X_test_scaled.shape}")

Data preprocessing complete!
Training set shape: (400, 4)
Testing set shape: (100, 4)


In [ ]:
# Cell 4: Train k-NN and Logistic Regression classifiers
# 1. Train k-NN Model
knn_model = KNeighborsClassifier(n_neighbors=5)
knn_model.fit(X_train_scaled, y_train)
knn_preds = knn_model.predict(X_test_scaled)

# 2. Train Logistic Regression Model
log_model = LogisticRegression(random_state=42)
log_model.fit(X_train_scaled, y_train)
log_preds = log_model.predict(X_test_scaled)

# 3. Print Accuracy Comparison
knn_acc = accuracy_score(y_test, knn_preds)
log_acc = accuracy_score(y_test, log_preds)

print("=== MODEL EVALUATION RESULTS ===")
print(f"k-Nearest Neighbors (k-NN) Accuracy: {knn_acc * 100:.2f}%")
print(f"Logistic Regression Accuracy:        {log_acc * 100:.2f}%")
print("\nClassification Report (Logistic Regression):")
print(classification_report(y_test, log_preds))

=== MODEL EVALUATION RESULTS ===
k-Nearest Neighbors (k-NN) Accuracy: 98.00%
Logistic Regression Accuracy:        98.00%

Classification Report (Logistic Regression):
              precision    recall  f1-score   support

           0       1.00      0.60      0.75         5
           1       0.98      1.00      0.99        95

    accuracy                           0.98       100
   macro avg       0.99      0.80      0.87       100
weighted avg       0.98      0.98      0.98       100



In [ ]:
# Cell 5: Interactive Prediction Interface (Frontend inside Jupyter)

def predict_student_risk():
    print("--- STUDENT RISK PREDICTION SYSTEM ---")
    
    # Take user input for a new student
    attendance_input = float(input("Enter Attendance Rate (0-100%): "))
    assignment_input = float(input("Enter Assignment Score (0-100%): "))
    quiz_input = float(input("Enter Quiz Score (0-100%): "))
    study_input = float(input("Enter Weekly Study Hours (0-30 hrs): "))
    
    # 1. Format input into DataFrame with matching feature names (fixes UserWarning)
    new_student = pd.DataFrame([[attendance_input, assignment_input, quiz_input, study_input]], 
                               columns=['Attendance_Rate', 'Assignment_Score', 'Quiz_Score', 'Study_Hours'])
    
    # 2. Scale features using trained scaler
    new_student_scaled = scaler.transform(new_student)
    
    # 3. Predict using Logistic Regression model
    prediction = log_model.predict(new_student_scaled)[0]
    probability = log_model.predict_proba(new_student_scaled)[0][1] * 100
    
    # 4. Display result
    print("\n=== PREDICTION RESULT ===")
    if prediction == 1:
        print(f"Status: SAFE / PASS (Confidence: {probability:.2f}%)")
        print("Recommendation: Student is performing well. Maintain current study routine.")
    else:
        print(f"Status: AT-RISK / FAIL (Confidence: {100 - probability:.2f}%)")
        print("Recommendation: Early intervention required! Schedule counseling and extra tutoring.")

    # 5. Display Warning & Disclaimer
    print("\n> DISCLAIMER / WARNING:")
    print("> 1. Predictions are based on standard statistical thresholds and synthetic training patterns.")
    print("> 2. Results should be used as an advisory guide by faculty, not as an automated grading decision.")

# Run the frontend interface
predict_student_risk()
    

--- STUDENT RISK PREDICTION SYSTEM ---


Enter Attendance Rate (0-100%):  85
Enter Assignment Score (0-100%):  90
Enter Quiz Score (0-100%):  80
Enter Weekly Study Hours (0-30 hrs):  15



=== PREDICTION RESULT ===
Status: SAFE / PASS (Confidence: 100.00%)
Recommendation: Student is performing well. Maintain current study routine.

> DISCLAIMER / WARNING:
> 1. Predictions are based on standard statistical thresholds and synthetic training patterns.
> 2. Results should be used as an advisory guide by faculty, not as an automated grading decision.
